# Screened B — training + GSM/MATH development evaluation

This updated notebook automatically completes **11 B training runs**, then
evaluates them on the same **500 GSM + 500 MATH development questions as C0/C1**.
The completed screened B pilot and its development results are reused.

Each full model trains on **1,000 selected examples from the 4,955-row pool**.
Original B and C0/C1 files are preserved. **No reserved final tests are run.**

Upload this updated notebook to your Sol home folder, open it there, and choose
**Run → Run All Cells**. Wait for the submission cell to return actual job IDs
and **`all_jobs_released: true`**; then you may disconnect and turn off your laptop.
If the earlier notebook already submitted B training, this version reuses it.


## 1. Install the included code

The training package is unchanged. The evaluation code is installed into a new
folder. Different existing files are preserved and cause a stop.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, stat, subprocess, uuid, zipfile

WORK = Path('/scratch/sshind38/capstone_c0_c1_work')
BROOT = Path('/scratch/sshind38/capstone_approach_b')
PYTHON = BROOT / 'venv_b_training_py312/bin/python'
if not WORK.is_dir() or not PYTHON.is_file():
    raise RuntimeError('Run this updated notebook on Sol with the existing project files and B environment.')
PACKAGE = WORK / 'B_Screened_4955_Controls_v1'
EVAL_PACKAGE = WORK / 'B_Screened_4955_Development_Evaluation_v1'
TRAINING_ENCODED = '...'
EVALUATION_ENCODED = '...'
def install_package(directory, encoded, wanted_sha256):
    payload = base64.b64decode(encoded, validate=True)
    if hashlib.sha256(payload).hexdigest() != wanted_sha256:
        raise RuntimeError('Embedded package damaged; use the original downloaded notebook.')
    if directory.is_symlink():
        raise RuntimeError('Package directory is a symlink; preserved: ' + str(directory))
    directory.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            rel = Path(member.filename)
            if rel.is_absolute() or '..' in rel.parts or member.is_dir() or stat.S_ISLNK(member.external_attr >> 16):
                raise RuntimeError('Invalid embedded package member')
            destination = directory / rel
            if destination.is_symlink() or not destination.resolve().is_relative_to(directory.resolve()):
                raise RuntimeError('Unsafe package destination')
            content = archive.read(member)
            if destination.exists():
                if destination.read_bytes() != content:
                    raise RuntimeError('Different existing file preserved: ' + str(destination))
            else:
                temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
                with temporary.open('xb') as stream:
                    stream.write(content); stream.flush(); os.fsync(stream.fileno())
                os.link(temporary, destination)
install_package(PACKAGE, TRAINING_ENCODED, 'c2124872dc477048d753ab4b0a75969dfd31c5a84f4cdda2b09b8f41a4065615')
install_package(EVAL_PACKAGE, EVALUATION_ENCODED, 'e01874c02e41f04da0116e8f421d9a4bad4aa262daed33d5ec8db7ab44c767df')
RUNNER = EVAL_PACKAGE / 'evaluate_b.py'
def run_b(action, extra=()):
    command = [str(PYTHON), '-u', '-B', str(RUNNER), action,
               '--work', str(WORK), '--b-root', str(BROOT), *map(str, extra)]
    return subprocess.run(command, cwd=WORK, check=True)
print('Training code preserved:', PACKAGE)
print('Added evaluation code:', EVAL_PACKAGE)
print('Plan: 11 new B training runs, then 22 new development files; reuse the pilot and its 2 files.')


## 2. Submit training followed by development evaluation

CPU preflight validates the bank, selected data, parents, existing pilot, native
evaluator/parser and frozen development questions before submission. The default
is **two serial training allocations followed by three serial evaluation
allocations**, each with one A100 and a four-hour limit. These are limits, not
runtime estimates. Queue time is additional.

Wait until this cell returns and displays **evaluation job IDs** and
**`all_jobs_released: true`**. Then Slurm continues both stages independently of
your laptop or notebook. Existing active training/evaluation jobs are reused.


In [ ]:
ACCOUNT = 'class_fse57081354fall2026'
PARTITION = 'public'
QOS = 'class'
TRAINING_ALLOCATIONS = 2
EVALUATION_ALLOCATIONS = 3
WORKER_SECONDS = 12600
RESOURCES = ['--account', ACCOUNT, '--partition', PARTITION, '--qos', QOS,
             '--training-allocations', str(TRAINING_ALLOCATIONS),
             '--eval-allocations', str(EVALUATION_ALLOCATIONS),
             '--worker-seconds', str(WORKER_SECONDS)]
run_b('submit', RESOURCES)
print('Submission cell finished. Confirm real job IDs and all_jobs_released: true above.')
print('Then you can close the laptop and disconnect; training and development evaluation remain under Slurm.')


## 3. Status — rerun after reconnecting

Shows both training and evaluation jobs, completed model counts, and recent
stage output. Evaluation jobs pending on a dependency are expected while
training runs. Training must reach **12/12**; development evaluation must then
reach **12/12 models and 24/24 files**. Counts include the reused pilot.


In [ ]:
run_b('status')

## 4. Progress and exact locations

Training and evaluation progress are separate. During evaluation the stage log
shows question counts such as `16/500 done`; the file count advances after the
whole 500-question file passes validation.


In [ ]:
for title, directory in [('TRAINING', PACKAGE), ('DEVELOPMENT EVALUATION', EVAL_PACKAGE)]:
    location_path = directory / 'RUN_LOCATION.json'
    print()
    print(title)
    if not location_path.is_file():
        print('No recorded submission yet. Check the submission cell output.')
        continue
    location = json.loads(location_path.read_text())
    control = Path(location['control_directory'])
    print('Exact output directory:', control)
    status_path = control / 'STATUS.json'
    if status_path.is_file():
        current = json.loads(status_path.read_text())
        for key in ['status', 'B_screened_complete_of_12', 'new_B_training_complete_of_11',
                    'B_screened_models_evaluated_of_12', 'completed_evaluation_files',
                    'new_evaluation_files_complete_of_22', 'completed_questions']:
            if key in current:
                print(key + ':', current[key])
    print('Job IDs:', ', '.join(j['job_id'] for j in location.get('jobs', [])))
    if title == 'DEVELOPMENT EVALUATION':
        print('Completion marker:', control / 'EVALUATION_COMPLETE.json')
        print('Summary CSV:', control / 'B_SCREENED_SUMMARY.csv')


## 5. Download the combined results after both stages finish

Rerun this cell after evaluation reaches **12/12 models and 24/24 files**.
It validates completion and copies the combined handoff ZIP into your **Sol home
folder**. The ZIP includes raw predictions, configs, metrics, a B summary CSV,
training receipts and the checkpoint index. Adapter weights remain on Sol.


In [ ]:
location_path = EVAL_PACKAGE / 'RUN_LOCATION.json'
if not location_path.is_file():
    print('No evaluation submission recorded yet. Check the submission output.')
else:
    CONTROL = Path(json.loads(location_path.read_text())['control_directory'])
    marker = CONTROL / 'EVALUATION_COMPLETE.json'
    if not marker.is_file():
        status_path = CONTROL / 'STATUS.json'
        current = json.loads(status_path.read_text()) if status_path.is_file() else {}
        print('Status:', current.get('status', 'not started'))
        print('B models evaluated:', str(current.get('B_screened_models_evaluated_of_12', 0)) + '/12')
        print('Development result files:', str(current.get('completed_evaluation_files', 0)) + '/24')
        print('Rerun the Status cell for jobs and logs; return here after verified completion.')
    else:
        run_b('status')  # Deep validation also creates/validates the combined handoff.
        source = CONTROL / ('B_Screened_4955_Training_and_Evaluation_Handoff_' + CONTROL.name + '.zip')
        content = source.read_bytes()
        destination = Path.home() / source.name
        if destination.exists() and destination.read_bytes() != content:
            destination = Path.home() / (source.stem + '_' + hashlib.sha256(content).hexdigest()[:12] + '.zip')
        if destination.exists():
            if destination.read_bytes() != content:
                raise RuntimeError('Different existing download preserved: ' + str(destination))
        else:
            with destination.open('xb') as stream:
                stream.write(content); stream.flush(); os.fsync(stream.fileno())
        print('Verified: screened B training 12/12, development evaluation 12/12 models and 24/24 files.')
        print('Download this combined results ZIP from your Sol HOME folder:')
        print(destination)
        print('Use the screened B summary and predictions with screened C0/C1 for the matched comparison.')


### Recovery and interpretation

If the allocation chain finishes with incomplete counts, rerun the submission
cell to continue; validated completed work is reused. Compatible partial
evaluation files resume by question ID. Existing partial attempts remain
preserved. Integrity failures and uncertain sbatch outcomes require review;
the code blocks blind resubmission or silent changes to the scientific inputs.
Unused queued allocations in this submission are cancelled when no work remains.

No new C0/C1 evaluation is needed here: their development results already exist.
The matched comparison uses **screened B versus screened C0/C1**, with legacy B
kept separate. A full comparative analysis follows once these new B results exist.
Inherited parent/development exposure and the unresolved historical screening
audit remain disclosed. C1's existing exploratory interpretation and diagnostic
gate are unchanged. **No reserved final tests are run.**
